## 01 - feature engineering
Feature engineering for fraud detection. Build spatial, temporal rolling-window and profile features from `clean_fraud.csv`; output is used by CatBoost, calibration and IF.

**Output**: `data/features_processed.csv` (includes is_online, distance_to_merchant, effective_distance, hour_of_day, day_of_week, tx_count_1h, tx_count_24h, amt_sum_24h, customer_age, etc.)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd()  # run from project root
DATA_RAW = ROOT / "src" / "data" / "processed" / "clean_fraud.csv"
OUT_DIR = ROOT / "fraud_detection" / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_RAW)
df["transaction_datetime"] = pd.to_datetime(df["transaction_datetime"])
df["customer_dob"] = pd.to_datetime(df["customer_dob"], errors="coerce")
print("Shape:", df.shape)
df.head(2)

Shape: (555719, 22)


,transaction_datetime,customer_id_number,merchant_name,merchant_category,transaction_amount,customer_first_name,customer_last_name,customer_gender,customer_street,customer_city,...,customer_latitude,customer_longitude,customer_city_population,customer_job_title,customer_dob,transaction_id,transaction_unix_time,merchant_latitude,merchant_longitude,is_fraud
0,2020-06-21 12:14:00,2.291160e+15,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,...,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,2020-06-21 12:14:00,3.573030e+15,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,...,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0


Spatial features: is_online, distance_to_merchant, effective_distance

In [2]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    return R * c

cat_col = "merchant_category"
df["is_online"] = (df[cat_col].fillna("").str.endswith("_net")).astype(int)

lat_c, lon_c = df["customer_latitude"], df["customer_longitude"]
lat_m, lon_m = df["merchant_latitude"], df["merchant_longitude"]
valid = lat_c.notna() & lon_c.notna() & lat_m.notna() & lon_m.notna()
df["distance_to_merchant"] = np.nan
df.loc[valid, "distance_to_merchant"] = haversine_km(
    lat_c[valid].values, lon_c[valid].values,
    lat_m[valid].values, lon_m[valid].values
)
df["effective_distance"] = np.where(df["is_online"] == 1, 0.0, df["distance_to_merchant"])
df["effective_distance"] = df["effective_distance"].fillna(-1)  # fill missing with -1
print(df[["is_online", "distance_to_merchant", "effective_distance"]].describe())

           is_online  distance_to_merchant  effective_distance
count  555719.000000         555719.000000       555719.000000
mean        0.159383             76.104902           63.961194
std         0.366033             29.117079           38.587591
min         0.000000              0.123883            0.000000
25%         0.000000             55.286255           36.218022
50%         0.000000             78.179517           70.390865
75%         0.000000             98.520760           94.345964
max         1.000000            150.922504          150.922504


Time features: hour_of_day, day_of_week

In [3]:
ts = df["transaction_datetime"]
df["hour_of_day"] = ts.dt.hour
df["day_of_week"] = ts.dt.dayofweek
print(df[["hour_of_day", "day_of_week"]].describe())

         hour_of_day    day_of_week
count  555719.000000  555719.000000
mean       12.809062       2.726779
std         6.810924       2.178681
min         0.000000       0.000000
25%         7.000000       1.000000
50%        14.000000       2.000000
75%        19.000000       5.000000
max        23.000000       6.000000


Rolling-window features (by card + time): tx_count_1h, tx_count_24h, amt_sum_24h

In [4]:
card_col = "customer_id_number"
df = df.sort_values([card_col, "transaction_datetime"]).reset_index(drop=True)

def rolling_agg(g, window_hours=24):
    t = g["transaction_datetime"].values.astype("datetime64[s]")
    t_sec = t.astype("int64")  # seconds since epoch
    amt = g["transaction_amount"].values.astype(float)
    n = len(t)
    if n == 0:
        return pd.DataFrame({"tx_count_1h": [], "tx_count_24h": [], "amt_sum_24h": []}, index=g.index)
    dt = t_sec[:, None] - t_sec[None, :]  # (n,n): dt[i,j] = t[i]-t[j] in seconds
    past_only = dt >= 0  # j is in the past or same time as i
    within_1h = (dt <= 3600) & (dt >= 0)
    within_24h = (dt <= window_hours * 3600) & (dt >= 0)
    np.fill_diagonal(within_1h, False)
    np.fill_diagonal(within_24h, False)
    cnt_1h = within_1h.sum(axis=1)
    cnt_24h = within_24h.sum(axis=1)
    sum_24h = (within_24h.astype(float) * amt).sum(axis=1)
    return pd.DataFrame({"tx_count_1h": cnt_1h, "tx_count_24h": cnt_24h, "amt_sum_24h": sum_24h}, index=g.index)

print("Computing velocity features (may take a few minutes)...")
rolled = df.groupby(card_col, group_keys=False).apply(rolling_agg)
df["tx_count_1h"] = rolled["tx_count_1h"].values
df["tx_count_24h"] = rolled["tx_count_24h"].values
df["amt_sum_24h"] = rolled["amt_sum_24h"].values
print(df[["tx_count_1h", "tx_count_24h", "amt_sum_24h"]].describe())

Computing velocity features (may take a few minutes)...
         tx_count_1h   tx_count_24h    amt_sum_24h
count  555719.000000  555719.000000  555719.000000
mean        0.234221       4.766598     331.209018
std         0.518143       3.692574     453.059945
min         0.000000       0.000000       0.000000
25%         0.000000       2.000000      90.225000
50%         0.000000       4.000000     217.730000
75%         0.000000       7.000000     420.900000
max         8.000000      37.000000   23136.720000


/var/folders/fj/8yv81kvs0q37tmhv5tn6_shr0000gn/T/ipykernel_8608/1860101127.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rolled = df.groupby(card_col, group_keys=False).apply(rolling_agg)


User profile: customer_age

In [5]:
df["customer_age"] = (df["transaction_datetime"] - df["customer_dob"]).dt.days / 365.25
df["customer_age"] = df["customer_age"].clip(lower=0, upper=120)
df["customer_age"] = df["customer_age"].fillna(-1)  # missing
print(df["customer_age"].describe())

count    555719.000000
mean         46.887987
std          17.431204
min          15.392197
25%          33.448323
50%          44.908966
75%          58.056126
max          96.169747
Name: customer_age, dtype: float64


Save feature table

In [6]:
FEATURE_PATH = OUT_DIR / "features_processed.csv"
df.to_csv(FEATURE_PATH, index=False)
print(f"Saved to {FEATURE_PATH}")
print("Columns:", list(df.columns))

Saved to /Users/zhumiban/Desktop/agent_bank/fraud_detection/data/features_processed.csv
Columns: ['transaction_datetime', 'customer_id_number', 'merchant_name', 'merchant_category', 'transaction_amount', 'customer_first_name', 'customer_last_name', 'customer_gender', 'customer_street', 'customer_city', 'customer_state', 'customer_zip', 'customer_latitude', 'customer_longitude', 'customer_city_population', 'customer_job_title', 'customer_dob', 'transaction_id', 'transaction_unix_time', 'merchant_latitude', 'merchant_longitude', 'is_fraud', 'is_online', 'distance_to_merchant', 'effective_distance', 'hour_of_day', 'day_of_week', 'tx_count_1h', 'tx_count_24h', 'amt_sum_24h', 'customer_age']
